# probe: read-path sanity on working+input mounts


In [ ]:
import glob, hashlib, os, time
def t(label, fn):
    t0=time.monotonic()
    try:
        out=fn()
        print(f"{label} OK {time.monotonic()-t0:.1f}s {out}", flush=True)
    except Exception as e:
        print(f"{label} FAIL {time.monotonic()-t0:.1f}s {type(e).__name__}:{e}", flush=True)
t("write-1MB-working", lambda: open("/kaggle/working/r.bin","wb").write(os.urandom(1<<20)))
t("readback-1MB-working", lambda: len(open("/kaggle/working/r.bin","rb").read()))
t("glob-input", lambda: len(glob.glob("/kaggle/input/**/qwen38-vllm0272-cu130-wheelhouse-v1/**/requirements-qwen38-runtime.txt", recursive=True)))
def read64():
    p=glob.glob("/kaggle/input/**/qwen38-vllm0272-cu130-wheelhouse-v1/**/wheels/*.whl", recursive=True)[0]
    h=hashlib.sha256(); n=0
    t0=time.monotonic()
    with open(p,"rb") as f:
        while n < (64<<20):
            if time.monotonic()-t0 > 100: return f"timeout-bytes={n}"
            ch=f.read(1<<20)
            if not ch: break
            h.update(ch); n+=len(ch)
    return f"bytes={n} sha={h.hexdigest()[:12]}"
t("read64-input-whl", read64)
print("PROBE DONE", flush=True)
